# RCDM Training on Google Colab

This notebook trains the RCDM (Representation-Conditioned Diffusion Model) on Tiny ImageNet using Google Colab's free GPU.

## Setup Steps:
1. **Runtime → Change runtime type → GPU (T4 recommended)**
2. Run cells top to bottom
3. When prompted, sign in to Google Drive (for saving checkpoints)
4. Upload `train_reps.pt` to Google Drive beforehand for fastest data access

## Repo:
[github.com/SeverinLe/master_implementation](https://github.com/SeverinLe/master_implementation)

## 1. Check GPU Availability

In [1]:
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("⚠️ No GPU detected! Go to Runtime → Change runtime type → GPU")

PyTorch version: 2.10.0+cu128
CUDA available: True
GPU: NVIDIA H100 80GB HBM3
Memory: 85.02 GB


## 2. Mount Google Drive (Optional - for storing checkpoints)

In [2]:
from google.colab import drive
drive.mount('/content/drive')

# Create checkpoint directory in Drive
!mkdir -p /content/drive/MyDrive/rcdm_checkpoints

Mounted at /content/drive


## 3. Clone Code from GitHub

In [ ]:
import os

REPO_URL = "https://github.com/SeverinLe/master_implementation.git"
REPO_DIR = "/content/master_implementation"

if os.path.exists(REPO_DIR):
    # Already cloned — just pull latest changes
    print("Repo already exists, pulling latest changes...")
    !git -C {REPO_DIR} pull
else:
    print(f"Cloning {REPO_URL} ...")
    !git clone {REPO_URL} {REPO_DIR}

%cd {REPO_DIR}

# Confirm structure
print("\nProject structure:")
for item in sorted(os.listdir(REPO_DIR)):
    print(f"  {item}")

Upload your zipped project folder...


> **Tip:** If you push new changes from Cursor, re-run the cell above to `git pull` them into Colab without restarting the runtime.

In [ ]:
# Verify the clone contains all expected modules
!ls rcdm/ scripts/ guided_diffusion/guided_diffusion/

## 4. Install Dependencies

In [4]:
# Install the guided_diffusion package in editable mode
# setup.py is at guided_diffusion/setup.py (relative to repo root)
!pip install -e guided_diffusion/ -q

# Other dependencies
!pip install "blobfile>=1.0.5" tqdm pillow torchvision -q

print("✓ Dependencies installed")

ERROR: ./guided_diffusion is not a valid editable requirement. It should either be a path to a local project or a VCS URL (beginning with bzr+http, bzr+https, bzr+ssh, bzr+sftp, bzr+ftp, bzr+lp, bzr+file, git+http, git+https, git+ssh, git+git, git+file, hg+file, hg+http, hg+https, hg+ssh, hg+static-http, svn+ssh, svn+http, svn+https, svn+svn, svn+file).
✓ Dependencies installed


## 5. Get Training Data

`train_reps.pt` (~826MB) is not stored in GitHub (too large). Two options:

- **Option A (recommended):** Upload it to Google Drive once, then copy with the cell below — fast and persists across sessions.
- **Option B:** Upload directly to Colab — works but takes longer and is lost when the session ends.

In [ ]:
import os

DATA_DIR = "data/tiny-imagenet-200"
DATA_FILE = f"{DATA_DIR}/train_reps.pt"
DRIVE_FILE = "/content/drive/MyDrive/train_reps.pt"

os.makedirs(DATA_DIR, exist_ok=True)

# Option A (recommended): copy from Google Drive
if os.path.exists(DRIVE_FILE):
    print("Copying train_reps.pt from Google Drive...")
    !cp {DRIVE_FILE} {DATA_FILE}
    print("✓ Copied from Drive")

# Option B: upload directly
elif not os.path.exists(DATA_FILE):
    from google.colab import files
    print("train_reps.pt not found in Drive. Upload it directly (826MB)...")
    uploaded = files.upload()
    !mv train_reps.pt {DATA_FILE}
    print("✓ Uploaded directly")

else:
    print("✓ train_reps.pt already present")

print(f"File size: {os.path.getsize(DATA_FILE) / 1e9:.2f} GB")

In [ ]:
# To save train_reps.pt to your Drive for future sessions (run once after uploading):
# !cp data/tiny-imagenet-200/train_reps.pt /content/drive/MyDrive/train_reps.pt

## 6. Verify Setup

In [ ]:
import os
import sys

# Add to Python path
sys.path.insert(0, '/content/master_implementation')

# Test imports
from guided_diffusion.script_util import create_model_and_diffusion
from rcdm.dataset import RepresentationDataset
from rcdm.encoder import load_encoder

# Verify imports are callable
print(f"✓ guided_diffusion: {create_model_and_diffusion.__name__}")
print(f"✓ rcdm.dataset: {RepresentationDataset.__name__}")
print(f"✓ rcdm.encoder: {load_encoder.__name__}")

# Check data file exists
data_path = '/content/master_implementation/data/tiny-imagenet-200/train_reps.pt'
assert os.path.exists(data_path), f"Data file not found: {data_path}"

print("\n✓ All imports successful")
print(f"✓ Data file found: {os.path.getsize(data_path) / 1e9:.2f} GB")

## 7. Start Training

Run your training script with GPU acceleration!

In [ ]:
# Training with GPU
!python scripts/train.py \
    --reps_file data/tiny-imagenet-200/train_reps.pt \
    --save_dir /content/drive/MyDrive/rcdm_checkpoints \
    --image_size 64 \
    --batch_size 128 \
    --lr 1e-4 \
    --total_steps 50000 \
    --save_interval 2000 \
    --log_interval 100 \
    --num_workers 4 \
    --device cuda

## 8. Monitor Training (Optional - Run in separate cell)

In [ ]:
# Check latest checkpoint
!ls -lh /content/drive/MyDrive/rcdm_checkpoints/

# Monitor GPU usage
!nvidia-smi

## 9. Generate Samples (After Training)

In [ ]:
# Load your sampling script here once you create it
# !python scripts/sampling.py --checkpoint /content/drive/MyDrive/rcdm_checkpoints/model_100000.pt

---

## Tips

| Topic | Guidance |
|-------|----------|
| **GPU runtime limit** | Free Colab = ~12 h. Save checkpoints frequently (`--save_interval 2000`). |
| **Batch size** | T4: use 64–128. A100/H100: up to 256. |
| **Checkpoints** | Saved to Google Drive — survive session restarts. |
| **Code changes** | Edit in Cursor → `git push` → re-run Cell 3 (`git pull`) in Colab. No restart needed. |
| **Monitor GPU** | Run `!nvidia-smi` in a spare cell at any time. |

## Resume after disconnection

Re-run cells 1–6 (clone/pull, install, data), then:

```python
!python scripts/train.py \
    --reps_file data/tiny-imagenet-200/train_reps.pt \
    --save_dir /content/drive/MyDrive/rcdm_checkpoints \
    --image_size 64 \
    --batch_size 128 \
    --lr 1e-4 \
    --total_steps 50000 \
    --save_interval 2000 \
    --log_interval 100 \
    --num_workers 4 \
    --resume /content/drive/MyDrive/rcdm_checkpoints/rcdm_stepXXXXXXX.pt \
    --device cuda
```